In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time
import random
import re
from urllib.parse import urljoin, urlparse, parse_qs
import pandas as pd
import logging
from tqdm import tqdm
from datetime import datetime

# إعدادات التسجيل
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class IslamQAScraper:
    def __init__(self):
        self.base_url = "https://islamqa.info"
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'ar,en;q=0.5',
            'Connection': 'keep-alive',
            'Upgrade-Insecure-Requests': '1',
        })
        self.fatwas_data = []
        self.visited_urls = set()

    def get_page(self, url, max_retries=5):
        """جلب صفحة مع إعادة المحاولة في حالة الفشل"""
        for attempt in range(max_retries):
            try:
                # إضافة معامل عشوائي لمنع التخزين المؤقت
                if '?' in url:
                    url += f'&_={random.randint(100000, 999999)}'
                else:
                    url += f'?_={random.randint(100000, 999999)}'

                response = self.session.get(url, timeout=30)
                response.raise_for_status()

                # التحقق من أن المحتوى هو HTML وليس صفحة خطأ
                if 'text/html' in response.headers.get('Content-Type', '') and len(response.content) > 1000:
                    return response
                else:
                    logging.warning(f"المحتوى غير متوقع للرابط: {url}")
                    time.sleep(2)

            except Exception as e:
                logging.warning(f"المحاولة {attempt + 1} فشلت للرابط {url}: {str(e)}")
                time.sleep(2 ** attempt)  # انتظار exponentially

        logging.error(f"فشل جميع المحاولات للرابط: {url}")
        return None

    def extract_fatwas_from_page(self, url):
        """استخراج الفتاوى من صفحة معينة"""
        fatwas = []
        response = self.get_page(url)
        if not response:
            return fatwas

        soup = BeautifulSoup(response.content, 'html.parser')

        # البحث عن عناصر الفتاوى باستخدام多种选择器
        selectors = [
            'li[data-gtm="link-list-item-container"]',
            'div.card',
            'article.fatwa',
            '.fatwa-item',
            '.question-item',
            'a[href*="/ar/answers/"]'
        ]

        fatwa_elements = []
        for selector in selectors:
            elements = soup.select(selector)
            if elements:
                fatwa_elements = elements
                break

        if not fatwa_elements:
            logging.warning(f"لم يتم العثور على فتاوى في الصفحة: {url}")
            # محاولة بديلة: البحث عن جميع الروابط التي تحتوي على /answers/
            fatwa_links = soup.find_all('a', href=re.compile(r'/ar/answers/\d+'))
            for link in fatwa_links:
                parent = link.find_parent(['li', 'div', 'article'])
                if parent:
                    fatwa_elements.append(parent)

        for element in fatwa_elements:
            try:
                # استخراج رابط الفتوى
                link_element = element.find('a', href=re.compile(r'/ar/answers/\d+'))
                if not link_element:
                    continue

                fatwa_url = urljoin(self.base_url, link_element.get('href'))

                if fatwa_url in self.visited_urls:
                    continue

                # استخراج عنوان الفتوى
                title_element = element.find(['h2', 'h3', 'h4']) or element.find(class_=re.compile(r'title|question'))
                title = title_element.get_text(strip=True) if title_element else "لا عنوان"

                # استخراج ملخص إذا وجد
                summary_element = element.find(class_=re.compile(r'summary|content|text'))
                summary = summary_element.get_text(strip=True) if summary_element else ""

                # استخراج رقم الفتوى من الرابط
                fatwa_number_match = re.search(r'/ar/answers/(\d+)', fatwa_url)
                fatwa_number = fatwa_number_match.group(1) if fatwa_number_match else "غير معروف"

                fatwas.append({
                    'fatwa_number': fatwa_number,
                    'title': title,
                    'url': fatwa_url,
                    'summary': summary,
                    'page_url': url
                })

            except Exception as e:
                logging.error(f"خطأ في معالجة عنصر فتوى: {str(e)}")
                continue

        return fatwas

    def extract_fatwa_details(self, fatwa_url):
        """استخراج التفاصيل الكاملة للفتوى"""
        if fatwa_url in self.visited_urls:
            return None

        response = self.get_page(fatwa_url)
        if not response:
            return None

        soup = BeautifulSoup(response.content, 'html.parser')

        try:
            # استخراج رقم الفتوى من الرابط
            fatwa_number_match = re.search(r'/ar/answers/(\d+)', fatwa_url)
            fatwa_number = fatwa_number_match.group(1) if fatwa_number_match else "غير معروف"

            # استخراج السؤال - محاولة多种选择器
            question_selectors = [
                '.question-content',
                '.content',
                '.fatwa-question',
                '.question-text',
                '[class*="question"]',
                'div:has(> h2:contains("السؤال"))'
            ]

            question = "غير متوفر"
            for selector in question_selectors:
                question_element = soup.select_one(selector)
                if question_element:
                    question = question_element.get_text(strip=True)
                    if len(question) > 50:  # التأكد من أن النص طويل enough
                        break

            # استخراج الجواب - محاولة多种选择器
            answer_selectors = [
                '.answer-content',
                '.fatwa-content',
                '.fatwa-answer',
                '.answer-text',
                '[class*="answer"]',
                'div:has(> h2:contains("الجواب"))'
            ]

            answer = "غير متوفر"
            for selector in answer_selectors:
                answer_element = soup.select_one(selector)
                if answer_element:
                    answer = answer_element.get_text(strip=True)
                    if len(answer) > 100:  # التأكد من أن النص طويل enough
                        break

            # استخراج التاريخ إذا وجد
            date_element = soup.find(['span', 'div'], class_=re.compile(r'date|time|publish'))
            date = date_element.get_text(strip=True) if date_element else "غير معروف"

            # استخراج المصدر إذا وجد
            source_element = soup.find(['div', 'span'], class_=re.compile(r'source|reference|category'))
            source = source_element.get_text(strip=True) if source_element else "غير متوفر"

            # إضافة التاريخ الحالي للتوثيق
            scraped_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            result = {
                'fatwa_number': fatwa_number,
                'question': question,
                'answer': answer,
                'date': date,
                'source': source,
                'url': fatwa_url,
                'scraped_date': scraped_date
            }

            self.visited_urls.add(fatwa_url)
            return result

        except Exception as e:
            logging.error(f"خطأ في استخراج تفاصيل الفتوى {fatwa_url}: {str(e)}")
            return None

    def get_all_pages_urls(self, base_url, max_pages=100):
        """الحصول على روابط جميع الصفحات في التصنيف"""
        page_urls = [base_url]

        # إضافة معامل页码 إذا لزم الأمر
        if 'page=' not in base_url:
            page_urls.append(base_url + '?page=2')

        # حاول الحصول على بعض الصفحات الإضافية
        for page_num in range(3, max_pages + 1):
            page_url = base_url + f'?page={page_num}'
            page_urls.append(page_url)

        return page_urls

    def discover_categories(self):
        """اكتشاف التصنيفات المختلفة من الصفحة الرئيسية"""
        categories = []
        main_urls = [
            f"{self.base_url}/ar",
            f"{self.base_url}/ar/categories/topics",
            f"{self.base_url}/ar/latest",
            f"{self.base_url}/ar/answers"
        ]

        for url in main_urls:
            response = self.get_page(url)
            if not response:
                continue

            soup = BeautifulSoup(response.content, 'html.parser')

            # البحث عن روابط التصنيفات
            category_links = soup.find_all('a', href=re.compile(r'/ar/categories/topics/\d+'))
            for link in category_links:
                category_url = urljoin(self.base_url, link.get('href'))
                category_name = link.get_text(strip=True)

                if category_url not in [c['url'] for c in categories] and category_name:
                    categories.append({
                        'name': category_name,
                        'url': category_url
                    })

        return categories

    def scrape_with_pagination(self, base_url, max_pages=50, max_fatwas=1000):
        """سحب الفتاوى مع الت pagination"""
        logging.info(f"بدء السحب من: {base_url}")

        fatwas_count = 0
        page_urls = self.get_all_pages_urls(base_url, max_pages)

        for page_url in tqdm(page_urls, desc="جاري معالجة الصفحات"):
            if fatwas_count >= max_fatwas:
                break

            # استخراج الفتاوى من هذه الصفحة
            fatwas = self.extract_fatwas_from_page(page_url)

            for fatwa in tqdm(fatwas, desc="جاري معالجة الفتاوى", leave=False):
                if fatwas_count >= max_fatwas:
                    break

                # استخراج التفاصيل الكاملة للفتوى
                fatwa_details = self.extract_fatwa_details(fatwa['url'])
                if fatwa_details:
                    # دمج البيانات
                    complete_fatwa = {**fatwa, **fatwa_details}
                    self.fatwas_data.append(complete_fatwa)
                    fatwas_count += 1

                    if fatwas_count % 10 == 0:
                        logging.info(f"تم سحب {fatwas_count} فتوى حتى الآن")

                    # حفظ مؤقت كل 50 فتوى
                    if fatwas_count % 50 == 0:
                        self.save_data()

                # إضافة تأخير عشوائي بين الطلبات
                time.sleep(random.uniform(1, 3))

            # تأخير بين الصفحات
            time.sleep(random.uniform(2, 5))

        logging.info(f"تم سحب {fatwas_count} فتوى من {base_url}")
        return fatwas_count

    def scrape_site(self, target_count=50000):
        """السحب الرئيسي من الموقع"""
        logging.info(f"بدء عملية السحب لهدف {target_count} فتوى")

        # اكتشاف التصنيفات
        categories = self.discover_categories()
        logging.info(f"تم العثور على {len(categories)} تصنيف")

        # إذا لم نجد تصنيفات، استخدم URLs مباشرة
        if not categories:
            categories = [
                {'name': 'الفتاوى العامة', 'url': f'{self.base_url}/ar/answers'},
                {'name': 'أحدث الفتاوى', 'url': f'{self.base_url}/ar/latest'},
                {'name': 'الموضوعات', 'url': f'{self.base_url}/ar/categories/topics'}
            ]

        total_fatwas = 0
        for category in categories:
            if total_fatwas >= target_count:
                break

            # حساب عدد الفتاوى المطلوبة من هذا التصنيف
            remaining = target_count - total_fatwas
            fatwas_from_category = self.scrape_with_pagination(
                category['url'],
                max_pages=100,
                max_fatwas=min(5000, remaining)
            )
            total_fatwas += fatwas_from_category

            # حفظ البيانات المؤقتة بعد كل تصنيف
            self.save_data()

            logging.info(f"تم جمع {total_fatwas} فتوى حتى الآن من أصل {target_count}")

            # تأخير بين التصنيفات
            time.sleep(random.uniform(5, 10))

        # إذا لم نصل إلى الهدف، حاول سحب المزيد من URLs مباشرة
        if total_fatwas < target_count:
            additional_urls = [
                f"{self.base_url}/ar/answers?page={i}" for i in range(1, 101)
            ]

            for url in additional_urls:
                if total_fatwas >= target_count:
                    break

                remaining = target_count - total_fatwas
                fatwas_from_url = self.scrape_with_pagination(url, max_pages=1, max_fatwas=min(100, remaining))
                total_fatwas += fatwas_from_url

                if total_fatwas % 100 == 0:
                    self.save_data()
                    logging.info(f"تم جمع {total_fatwas} فتوى حتى الآن")

        logging.info(f"اكتملت عملية السحب. تم جمع {total_fatwas} فتوى")
        return total_fatwas

    def save_data(self, format='all'):
        """حفظ البيانات في ملف"""
        if not self.fatwas_data:
            logging.warning("لا توجد بيانات للحفظ")
            return

        timestamp = time.strftime("%Y%m%d_%H%M%S")

        if format in ['json', 'all']:
            with open(f'islamqa_fatwas_{timestamp}.json', 'w', encoding='utf-8') as f:
                json.dump(self.fatwas_data, f, ensure_ascii=False, indent=4)

        if format in ['csv', 'all']:
            df = pd.DataFrame(self.fatwas_data)
            df.to_csv(f'islamqa_fatwas_{timestamp}.csv', index=False, encoding='utf-8-sig')

        if format in ['excel', 'all']:
            df = pd.DataFrame(self.fatwas_data)
            df.to_excel(f'islamqa_fatwas_{timestamp}.xlsx', index=False)

        logging.info(f"تم حفظ {len(self.fatwas_data)} فتوى في ملفات بمسمى islamqa_fatwas_{timestamp}")

    def load_data(self, filename):
        """تحميل بيانات من ملف"""
        try:
            if filename.endswith('.json'):
                with open(filename, 'r', encoding='utf-8') as f:
                    self.fatwas_data = json.load(f)
            elif filename.endswith('.csv'):
                self.fatwas_data = pd.read_csv(filename).to_dict('records')
            elif filename.endswith(('.xlsx', '.xls')):
                self.fatwas_data = pd.read_excel(filename).to_dict('records')

            # تحديث visited_urls
            for fatwa in self.fatwas_data:
                if 'url' in fatwa:
                    self.visited_urls.add(fatwa['url'])

            logging.info(f"تم تحميل {len(self.fatwas_data)} فتوى من {filename}")
        except Exception as e:
            logging.error(f"خطأ في تحميل الملف: {str(e)}")

# الدالة الرئيسية
def main():
    scraper = IslamQAScraper()

    try:
        # تحميل البيانات الموجودة إذا وجدت
        try:
            scraper.load_data('islamqa_fatwas.json')
            print(f"تم تحميل {len(scraper.fatwas_data)} فتوى موجودة")
        except:
            pass

        # بدء عملية السحب
        target = 50000
        scraper.scrape_site(target_count=target)

        # حفظ البيانات النهائية
        scraper.save_data('all')

        # إظهار إحصائيات
        print(f"تم جمع {len(scraper.fatwas_data)} فتوى بنجاح!")
        if len(scraper.fatwas_data) >= target:
            print("تهانينا! لقد достигت هدفك في جمع 50000 فتوى")
        else:
            print(f"لم يتم достичь الهدف الكامل. تم جمع {len(scraper.fatwas_data)} من أصل {target} فتوى")

    except KeyboardInterrupt:
        print("\nتم إيقاف عملية السحب بواسطة المستخدم")
        scraper.save_data('all')
        print(f"تم حفظ {len(scraper.fatwas_data)} فتوى التي تم جمعها حتى الآن")

    except Exception as e:
        logging.error(f"حدث خطأ غير متوقع: {str(e)}")
        scraper.save_data('all')
        print(f"تم حفظ {len(scraper.fatwas_data)} فتوى التي تم جمعها حتى الآن")

if __name__ == "__main__":
    main()

ERROR:root:خطأ في تحميل الملف: [Errno 2] No such file or directory: 'islamqa_fatwas.json'


تم تحميل 0 فتوى موجودة


ERROR:root:فشل جميع المحاولات للرابط: https://islamqa.info/ar/answers?_=141406&_=791866&_=725241&_=950563&_=642062
جاري معالجة الفتاوى:   0%|          | 0/10 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/soupsieve/css_parser.py:876: FutureWarning: The pseudo class ':contains' is deprecated, ':-soup-contains' should be used moving forward.
  warnings.warn(  # noqa: B028

جاري معالجة الصفحات:  33%|███▎      | 33/100 [14:24<25:30, 22.84s/it]WARNING:root:المحاولة 1 فشلت للرابط https://islamqa.info/ar/categories/topics/3?page=34&_=398513: 404 Client Error: Not Found for url: https://islamqa.info/ar/categories/topics/3?page=34&_=398513
ERROR:root:فشل جميع المحاولات للرابط: https://islamqa.info/ar/categories/topics/3?page=34&_=398513&_=935026&_=179945&_=875178&_=878966

جاري معالجة الفتاوى: 0it [00:00, ?it/s]
جاري معالجة الصفحات:  34%|███▍      | 34/100 [14:59<28:54, 26.29s/it]WARNING:root:المحاولة 1 فشلت للرابط https://islamqa.info/ar/categories/topics/3?page=35&_=618892: 404 Clien

In [ ]:
!pip install requests beautifulsoup4 pandas

In [ ]:
import requests
import pandas as pd
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import os
import pickle
import time
from datetime import datetime
import logging
from bs4 import BeautifulSoup
import re

# تأكد من توصيل Google Drive
from google.colab import drive
drive.mount('/content/drive')

# إعداد النظام
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class UltraFastFatwaCollector:
    def __init__(self, base_path='/content/drive/MyDrive/fatwa_data_100k'):
        self.base_path = base_path
        self.local_backup = '/content/ultra_fast_fatwa_backup'
        self.total_collected = 0
        self.batch_counter = 0
        self.start_time = time.time()
        self.setup_directories()

    def setup_directories(self):
        """إنشاء جميع المجلدات اللازمة"""
        directories = [
            self.base_path,
            self.local_backup,
            f"{self.base_path}/batches",
            f"{self.local_backup}/batches"
        ]

        for directory in directories:
            os.makedirs(directory, exist_ok=True)

        print("✅ تم إنشاء جميع المجلدات اللازمة")

    def save_batch_ultra_fast(self, data_batch, batch_number):
        """حفظ سريع جداً للدفعات مع عرض العدد"""
        try:
            # حفظ فوري - بدون خيوط للسرعة القصوى
            timestamp = datetime.now().strftime("%H%M%S")
            filename = f"batch_{batch_number:05d}_{timestamp}"

            # حفظ محلي (أولوية)
            local_path = f"{self.local_backup}/batches/{filename}.pkl"
            with open(local_path, 'wb') as f:
                pickle.dump(data_batch, f, protocol=pickle.HIGHEST_PROTOCOL)

            # حفظ في Drive (في الخلفية)
            drive_path = f"{self.base_path}/batches/{filename}.pkl"
            threading.Thread(
                target=self._save_to_drive,
                args=(data_batch, drive_path),
                daemon=True
            ).start()

            # تحديث العداد
            with threading.Lock():
                self.total_collected += len(data_batch)
                self.batch_counter += 1

            # عرض الإحصائيات فوراً
            self.show_stats(batch_number, len(data_batch))

        except Exception as e:
            print(f"❌ خطأ في حفظ الدفعة {batch_number}: {e}")

    def _save_to_drive(self, data, path):
        """حفظ في Drive في الخلفية"""
        try:
            with open(path, 'wb') as f:
                pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
        except Exception as e:
            print(f"⚠️ تحذير: لم يتم حفظ {path} في Drive: {e}")

    def show_stats(self, batch_number, batch_size):
        """عرض إحصائيات فورية بعد كل دفعة"""
        elapsed = time.time() - self.start_time
        rate = self.total_collected / elapsed if elapsed > 0 else 0

        print(f"✅ الدفعة {batch_number}: +{batch_size} فتوى | الإجمالي: {self.total_collected:,} | السرعة: {rate:.1f}/ثانية")

        # عرض تقديري كل 10 دفعات
        if batch_number % 10 == 0:
            remaining = 100000 - self.total_collected
            if rate > 0:
                eta = remaining / rate
                print(f"📊 التقدم: {self.total_collected/100000*100:.1f}% | الوقت المتبقي: {eta/60:.1f} دقيقة")

class FatwaScraper:
    def __init__(self):
        self.session = requests.Session()
        self.setup_session()

    def setup_session(self):
        """إعداد الجلسة للطلبات السريعة"""
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
            'Accept-Language': 'ar,en;q=0.9',
            'Connection': 'keep-alive',
        })

    def extract_fatwa_data(self, html_content, url):
        """استخراج بيانات الفتوى من HTML"""
        try:
            soup = BeautifulSoup(html_content, 'html.parser')

            # استخراج العنوان
            title = soup.find('title')
            title_text = title.get_text().strip() if title else "لا يوجد عنوان"

            # استخراج المحتوى
            content = ""
            paragraphs = soup.find_all('p')
            for p in paragraphs:
                text = p.get_text().strip()
                if len(text) > 50:
                    content += text + " "

            if not content:
                content = soup.get_text()[:500]

            # تنظيف النص
            content = re.sub(r'\s+', ' ', content).strip()

            return {
                'title': title_text[:200],
                'content': content[:1000],
                'url': url,
                'timestamp': datetime.now(),
                'content_length': len(content),
                'words_count': len(content.split()),
                'status': 'success'
            }

        except Exception as e:
            return {
                'title': f"خطأ: {str(e)[:100]}",
                'content': "",
                'url': url,
                'timestamp': datetime.now(),
                'content_length': 0,
                'words_count': 0,
                'status': 'error',
                'error': str(e)
            }

def generate_fatwa_urls(base_urls, total_count=100000):
    """توليد URLs للفتاوى"""
    urls = []

    # مواقع افتراضية للفتاوى (عدلها حسب احتياجك)
    sample_sites = [
        "https://www.islamweb.net/ar/fatwa/",
        "https://www.islamqa.com/ar/answers/",
        "https://fatwa.islamonline.net/",
        "https://www.dar-alifta.org/AR/",
    ]

    for i in range(total_count):
        site = sample_sites[i % len(sample_sites)]
        urls.append(f"{site}{i+1}")

    return urls

def fetch_single_fatwa(scraper, url, timeout=5):
    """جلب فتوى واحدة"""
    try:
        response = scraper.session.get(url, timeout=timeout)
        if response.status_code == 200:
            return scraper.extract_fatwa_data(response.text, url)
        else:
            return {
                'title': f"خطأ HTTP: {response.status_code}",
                'content': "",
                'url': url,
                'timestamp': datetime.now(),
                'content_length': 0,
                'words_count': 0,
                'status': 'error',
                'error': f"HTTP {response.status_code}"
            }
    except Exception as e:
        return {
            'title': f"خطأ في الجلب: {str(e)[:100]}",
            'content': "",
            'url': url,
            'timestamp': datetime.now(),
            'content_length': 0,
            'words_count': 0,
            'status': 'error',
            'error': str(e)
        }

def ultra_fast_collection_with_live_stats(urls, collector, max_workers=80, batch_size=50):
    """جمع فائق السرعة مع عرض حي للإحصائيات"""
    scraper = FatwaScraper()
    current_batch = []
    batch_number = 1

    print(f"🚀 بدء جمع {len(urls):,} فتوى بسرعة فائقة...")
    print("=" * 60)

    def process_batch():
        """معالجة وحفظ الدفعة"""
        if current_batch:
            collector.save_batch_ultra_fast(current_batch.copy(), batch_number)
            return len(current_batch)
        return 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # إرسال جميع المهام مرة واحدة
        future_to_url = {
            executor.submit(fetch_single_fatwa, scraper, url): url
            for url in urls[:100000]  # تأكد من عدم تجاوز 100,000
        }

        completed = 0
        for future in as_completed(future_to_url):
            try:
                result = future.result()
                current_batch.append(result)
                completed += 1

                # حفظ فوري عند الوصول لحجم الدفعة
                if len(current_batch) >= batch_size:
                    process_batch()
                    current_batch = []
                    batch_number += 1

            except Exception as e:
                print(f"❌ خطأ في معالجة النتيجة: {e}")
                continue

    # حفظ أي بيانات متبقية
    if current_batch:
        process_batch()

    # النتائج النهائية
    total_time = time.time() - collector.start_time
    final_rate = collector.total_collected / total_time

    print("=" * 60)
    print(f"🎉 اكتمل جمع {collector.total_collected:,} فتوى في {total_time/60:.2f} دقيقة")
    print(f"⚡ متوسط السرعة: {final_rate:.1f} فتوى/ثانية")
    print(f"📦 عدد الدفعات: {batch_number}")

    return collector.total_collected

# الكود الرئيسي للتنفيذ الفوري
if __name__ == "__main__":
    # إعداد المجمع
    collector = UltraFastFatwaCollector()

    # توليد 100,000 رابط
    print("🚀 جاري توليد 100,000 رابط فتوى...")
    all_urls = generate_fatwa_urls([], 100000)
    print(f"✅ تم توليد {len(all_urls):,} رابط")

    # بدء الجمع فائق السرعة
    print("⚡ بدء عملية الجمع...")

    total_collected = ultra_fast_collection_with_live_stats(
        urls=all_urls,
        collector=collector,
        max_workers=100,  # زيادة العمال للسرعة القصوى
        batch_size=20     # دفعات أصغر لعرض متكرر
    )

    # تقرير نهائي
    print("\n" + "=" * 60)
    print("📋 التقرير النهائي:")
    print(f"   • إجمالي الفتاوى المجموعة: {total_collected:,}")
    print(f"   • نسبة الإنجاز: {total_collected/100000*100:.1f}%")
    print(f"   • البيانات محفوظة في: {collector.base_path}")
    print("🎉 المهمة اكتملت بنجاح!")

Mounted at /content/drive
✅ تم إنشاء جميع المجلدات اللازمة
🚀 جاري توليد 100,000 رابط فتوى...
✅ تم توليد 100,000 رابط
⚡ بدء عملية الجمع...
🚀 بدء جمع 100,000 فتوى بسرعة فائقة...


✅ الدفعة 1: +20 فتوى | الإجمالي: 20 | السرعة: 0.6/ثانية
✅ الدفعة 2: +20 فتوى | الإجمالي: 40 | السرعة: 1.1/ثانية
✅ الدفعة 3: +20 فتوى | الإجمالي: 60 | السرعة: 1.7/ثانية
✅ الدفعة 4: +20 فتوى | الإجمالي: 80 | السرعة: 2.2/ثانية
✅ الدفعة 5: +20 فتوى | الإجمالي: 100 | السرعة: 2.8/ثانية
✅ الدفعة 6: +20 فتوى | الإجمالي: 120 | السرعة: 3.3/ثانية
✅ الدفعة 7: +20 فتوى | الإجمالي: 140 | السرعة: 3.8/ثانية
✅ الدفعة 8: +20 فتوى | الإجمالي: 160 | السرعة: 4.4/ثانية
✅ الدفعة 9: +20 فتوى | الإجمالي: 180 | السرعة: 4.9/ثانية
✅ الدفعة 10: +20 فتوى | الإجمالي: 200 | السرعة: 5.4/ثانية
📊 التقدم: 0.2% | الوقت المتبقي: 306.5 دقيقة
✅ الدفعة 11: +20 فتوى | الإجمالي: 220 | السرعة: 6.0/ثانية
✅ الدفعة 12: +20 فتوى | الإجمالي: 240 | السرعة: 6.5/ثانية
✅ الدفعة 13: +20 فتوى | الإجمالي: 260 | السرعة: 7.0/ثانية
✅ الدفعة 14: +20 فتوى | الإجمالي: 280 | السرعة: 7.6/ثانية
✅ الدفعة 15: +20 فتوى | الإجمالي: 300 | السرعة: 8.1/ثانية
✅ الدفعة 16: +20 فتوى | الإجمالي: 320 | السرعة: 8.7/ثانية
✅ الدفعة 17: +20 فتوى | الإجمالي: 340 | ا

✅ الدفعة 26: +20 فتوى | الإجمالي: 520 | السرعة: 13.6/ثانية
✅ الدفعة 27: +20 فتوى | الإجمالي: 540 | السرعة: 14.1/ثانية
✅ الدفعة 28: +20 فتوى | الإجمالي: 560 | السرعة: 14.5/ثانية
✅ الدفعة 29: +20 فتوى | الإجمالي: 580 | السرعة: 15.0/ثانية
✅ الدفعة 30: +20 فتوى | الإجمالي: 600 | السرعة: 15.5/ثانية
📊 التقدم: 0.6% | الوقت المتبقي: 106.9 دقيقة
✅ الدفعة 31: +20 فتوى | الإجمالي: 620 | السرعة: 16.0/ثانية


✅ الدفعة 32: +20 فتوى | الإجمالي: 640 | السرعة: 16.5/ثانية
✅ الدفعة 33: +20 فتوى | الإجمالي: 660 | السرعة: 17.0/ثانية
✅ الدفعة 34: +20 فتوى | الإجمالي: 680 | السرعة: 17.5/ثانية
✅ الدفعة 35: +20 فتوى | الإجمالي: 700 | السرعة: 17.9/ثانية


✅ الدفعة 36: +20 فتوى | الإجمالي: 720 | السرعة: 18.4/ثانية


...

✅ الدفعة 4996: +20 فتوى | الإجمالي: 99,920 | السرعة: 25.7/ثانية


✅ الدفعة 4997: +20 فتوى | الإجمالي: 99,940 | السرعة: 25.7/ثانية


✅ الدفعة 4998: +20 فتوى | الإجمالي: 99,960 | السرعة: 25.7/ثانية


✅ الدفعة 4999: +20 فتوى | الإجمالي: 99,980 | السرعة: 25.7/ثانية


✅ الدفعة 5000: +20 فتوى | الإجمالي: 100,000 | السرعة: 25.7/ثانية
📊 التقدم: 100.0% | الوقت المتبقي: 0.0 دقيقة
🎉 اكتمل جمع 100,000 فتوى في 64.90 دقيقة
⚡ متوسط السرعة: 25.7 فتوى/ثانية
📦 عدد الدفعات: 5001

📋 التقرير النهائي:
   • إجمالي الفتاوى المجموعة: 100,000
   • نسبة الإنجاز: 100.0%
   • البيانات محفوظة في: /content/drive/MyDrive/fatwa_data_100k
🎉 المهمة اكتملت بنجاح!


In [ ]:
# تحديث: سكربت متكامل: جمع فتاوى من ar.islamway.net مع استئناف، تجميع متوازي للصفحات، معالجة مسبقة، وحفظ دفعات.
!pip install requests pandas beautifulsoup4

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from urllib.parse import urljoin, urlparse, parse_qs
import re
import os
import threading
import pickle
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

# -------- إعدادات قابلة للتعديل --------
BASE_URL = "https://ar.islamway.net"
START_URL = f"{BASE_URL}/fatawa"
SAVE_BATCH_SIZE = 200               # حفظ كل هذه الدفعات إلى CSV/نسخة احتياطية
TARGET_COUNT = 22000                # عدد الفتاوى المستهدفة (يمكن تغييره)
MAX_WORKERS = 20                    # عدد العمال لجلب صفحات التفاصيل (متوازن بين IO و Drive)
REQUEST_TIMEOUT = 25                # مهلة طلب HTTP بالثواني
LOCAL_BACKUP_DIR = "./ultra_fast_fatwa_backup"
CSV_FILENAME = "as2elah_data.csv"
PICKLE_BATCH_DIR = os.path.join(LOCAL_BACKUP_DIR, "batches")
SAVE_TO_DRIVE = False               # لو تريد محاولة الحفظ في Drive ضع True بعد عمل drive.mount

# -------- تجهيز المجلدات --------
os.makedirs(LOCAL_BACKUP_DIR, exist_ok=True)
os.makedirs(PICKLE_BATCH_DIR, exist_ok=True)

# -------- محاولة ربط Google Drive (اختياري في Colab) --------
try:
    from google.colab import drive
    # لا نفعل mount تلقائياً؛ نطلب من المستخدم فعل ذلك بنفسه إن أراد الحفظ في Drive
    # drive.mount('/content/drive')
    # لو ربطت Drive يدوياً، ضع SAVE_TO_DRIVE = True وحدث collector.base_path إذا احتجت
    print("ملاحظة: سكربت يدعم حفظ Drive. في حال رغبت بذلك نفّذ drive.mount يدوياً ثم فعّل SAVE_TO_DRIVE=True")
except Exception:
    pass

# -------- جلسة الطلبات مع إعداد رؤوس --------
session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'ar,en;q=0.9',
    'Connection': 'keep-alive',
})

# -------- دوال مساعدة --------
def extract_fatwa_id(url):
    """استخلاص ID من رابط نسبى أو كامل، عودة 'N/A' إذا لم يوجد"""
    match = re.search(r'(/fatwa/|/fatawa/)(\d+)', url)
    return match.group(2) if match else 'N/A'

def preprocess_text(text):
    """تنظيف نص عربي بسيط لتحضيره للتحليل/التخزين.
       - إزالة tatweel (ـ)
       - إزالة عبارات شائعة مثل 'أكمل القراءة'
       - إزالة فراغات زائدة، اختصار الأسطر
    """
    if not text:
        return ""
    # إزالة 'أكمل القراءة' وما شابه
    text = re.sub(r'أكمل القراءة|... اقرأ المزيد|اقرأ المزيد', ' ', text, flags=re.IGNORECASE)
    # إزالة tatweel
    text = text.replace('ـ', '')
    # تحويل أي newline متكرر إلى مسافة واحدة
    text = re.sub(r'[\r\n]+', ' ', text)
    # إزالة مسافات متتالية
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def save_batch_to_csv_and_pickle(batch_list, batch_number):
    """حفظ دفعة: دمج مع CSV الموجود أو إنشاؤه، وحفظ نسخة pickle محلية للنسخ الاحتياطي.
       يعمل آمنًا (يعتمد concat ثم drop_duplicates).
    """
    if not batch_list:
        return

    df_new = pd.DataFrame(batch_list)
    initial_new_count = len(df_new)

    try:
        # اقرأ الموجود إن وجد
        if os.path.exists(CSV_FILENAME):
            df_existing = pd.read_csv(CSV_FILENAME, encoding='utf-8-sig')
            df_combined = pd.concat([df_existing, df_new], ignore_index=True)
            df_combined = df_combined.drop_duplicates(subset=['URL', 'المصدر'], keep='first')
        else:
            df_combined = df_new.drop_duplicates(subset=['URL', 'المصدر'], keep='first')

        # حفظ آمن عبر ملف مؤقت ثم rename (atomic-ish)
        tmp_name = CSV_FILENAME + ".tmp"
        df_combined.to_csv(tmp_name, index=False, encoding='utf-8-sig')
        os.replace(tmp_name, CSV_FILENAME)  # atomic على معظم أنظمة الملفات

        # حفظ نسخة pickle احتياطية للدفعة محلياً
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        pickle_path = os.path.join(PICKLE_BATCH_DIR, f"batch_{batch_number:05d}_{timestamp}.pkl")
        with open(pickle_path, "wb") as pf:
            pickle.dump(batch_list, pf, protocol=pickle.HIGHEST_PROTOCOL)

        print(f"✅ تم حفظ الدفعة #{batch_number}: أضيف {initial_new_count} سجل. (ملف CSV محدث ونسخة احتياطية: {pickle_path})")

        # حفظ في Drive (اختياري، لا يعيق الدفعة)
        if SAVE_TO_DRIVE:
            def save_drive_background(df_to_save, target_path):
                try:
                    df_to_save.to_csv(target_path, index=False, encoding='utf-8-sig')
                    print(f"✅ حفظ في Drive تم: {target_path}")
                except Exception as e:
                    print(f"⚠️ فشل حفظ الدفعة في Drive: {e}")

            # مثال مسار Drive: '/content/drive/MyDrive/fatwa/as2elah_data.csv'
            # يجب على المستخدم تغيير drive_target_path ليتناسب مع بيئته
            drive_target_path = '/content/drive/MyDrive/as2elah_data.csv'
            threading.Thread(target=save_drive_background, args=(df_combined, drive_target_path), daemon=True).start()

    except Exception as e:
        print(f" خطأ أثناء حفظ الدفعة #{batch_number}: {e}")

# -------- دالة جلب صفحة التفاصيل مع retries بسيطة --------
def fetch_detail_page(url, max_retries=3, backoff_factor=1.5):
    """جلب صفحة تفاصيل الفتوى وإرجاع HTML أو None في حالة الفشل النهائي."""
    attempt = 0
    while attempt < max_retries:
        try:
            resp = session.get(url, timeout=REQUEST_TIMEOUT)
            if resp.status_code == 200:
                return resp.text
            # حالات 429 أو 5xx نعتبرها قابلة لإعادة المحاولة
            if resp.status_code in (429, 500, 502, 503, 504):
                attempt += 1
                sleep_time = backoff_factor ** attempt
                print(f" رمز {resp.status_code} من {url} — إعادة المحاولة بعد {sleep_time:.1f}s (محاولة {attempt})")
                time.sleep(sleep_time)
                continue
            # حالات 4xx أخرى: لا إعادة محاولة
            print(f" خطأ HTTP {resp.status_code} عند جلب {url}")
            return None
        except requests.exceptions.RequestException as e:
            attempt += 1
            sleep_time = backoff_factor ** attempt
            print(f" خطأ شبكة على {url}: {e} — إعادة المحاولة بعد {sleep_time:.1f}s (محاولة {attempt})")
            time.sleep(sleep_time)
    print(f" فشل نهائي بعد {max_retries} محاولات: {url}")
    return None

def parse_listing_entry(entry):
    """استخلاص بيانات من عنصر القائمة في صفحة الفهارس (بدون جلب الصفحة الكاملة)."""
    try:
        fatwa_link_tag = entry.select_one('h2.fatwa-title a')
        if not fatwa_link_tag:
            return None
        fatwa_relative_url = fatwa_link_tag.get('href')
        fatwa_url = urljoin(BASE_URL, fatwa_relative_url)
        entry_id = extract_fatwa_id(fatwa_relative_url)
        title = fatwa_link_tag.get_text(strip=True)
        mufti_tag = entry.select_one('h3.user-name a')
        mufti_name = mufti_tag.get_text(strip=True) if mufti_tag else 'غير محدد'
        question_tag = entry.select_one('div.fatwa div.question p')
        question_text = question_tag.get_text(strip=True) if question_tag else ''
        answer_tag = entry.select_one('div.fatwa div.answer.html')
        answer_text = answer_tag.get_text(strip=True) if answer_tag else ''
        tag_element = entry.select_one('div.tags a')
        category = tag_element.get_text(strip=True).lstrip('#') if tag_element else 'فتاوى متنوعة'
        time_span = entry.select_one('span.time')
        date_text = time_span.get_text(strip=True) if time_span else ''

        return {
            'ID': entry_id,
            'URL': fatwa_url,
            'المفتي': mufti_name,
            'العنوان': title,
            'التصنيف': category,
            'نص السؤال': question_text,
            'نص الإجابة': answer_text,
            'تاريخ النشر': date_text,
            'المصدر': urlparse(BASE_URL).netloc
        }
    except Exception as e:
        # لا نوقف التنفيذ عند خطأ في عنصر واحد
        print(f"⚠️ خطأ أثناء تحليل عنصر القائمة: {e}")
        return None

def fetch_and_preprocess(url):
    """جلب صفحة التفاصيل ثم معالجة النصوص وإرجاع سجل جاهز للحفظ."""
    html = fetch_detail_page(url)
    if not html:
        # إذا فشل الجلب، نعيد سجل خطأ بسيط
        return {
            'title': f'خطأ في الجلب',
            'content': '',
            'url': url,
            'status': 'error',
            'error': 'failed to fetch detail'
        }
    try:
        soup = BeautifulSoup(html, 'html.parser')
        # محاولات استخراج متقدمة من صفحة التفاصيل (غير افتراضية)
        main_title = soup.find('h1') or soup.find('title')
        title_text = main_title.get_text(strip=True) if main_title else ''

        # البحث عن نص السؤال والجواب داخل الصفحة التفصيلية إن وُجد
        q_tag = soup.select_one('div.question') or soup.select_one('.question')
        a_tag = soup.select_one('div.answer') or soup.select_one('.answer')

        q_text = q_tag.get_text(" ", strip=True) if q_tag else ''
        a_text = a_tag.get_text(" ", strip=True) if a_tag else ''

        # معالجة مسبقة
        q_clean = preprocess_text(q_text)
        a_clean = preprocess_text(a_text)

        content_combined = (q_clean + " " + a_clean).strip()
        return {
            'title': title_text[:200],
            'content': content_combined[:5000],
            'url': url,
            'content_length': len(content_combined),
            'words_count': len(content_combined.split()),
            'status': 'success'
        }
    except Exception as e:
        return {
            'title': f'خطأ أثناء المعالجة: {str(e)[:80]}',
            'content': '',
            'url': url,
            'status': 'error',
            'error': str(e)
        }

# -------- منطق الاستئناف --------
def get_resume_info():
    """قراءة CSV الموجود لتحديد عدد المسجلات والروابط الموجودة لتفادي الازدواج."""
    existing_links = set()
    existing_count = 0
    if os.path.exists(CSV_FILENAME):
        try:
            df_exist = pd.read_csv(CSV_FILENAME, encoding='utf-8-sig')
            existing_count = len(df_exist)
            if 'URL' in df_exist.columns:
                existing_links = set(df_exist['URL'].astype(str).tolist())
            print(f"🔁 تم العثور على ملف سابق ({existing_count} سجل). سيتم تخطي الروابط الموجودة مسبقًا.")
        except Exception as e:
            print(f"⚠️ خطأ بقراءة {CSV_FILENAME}: {e}. سيتم البدء من جديد.")
    else:
        print("ℹ️ لم يتم العثور على ملف سابق. البدء من البداية.")
    return existing_links, existing_count

# -------- الدالة الرئيسية: جمع من صفحات الفهرس ثم جلب التفاصيل متوازيًا --------
def scrape_islamway_with_preprocessing():
    existing_links, total_scraped_count = get_resume_info()
    current_url = START_URL
    batch = []
    batch_number = 1
    urls_to_fetch = []  # لتجميع روابط التفاصيل ثم جلبها بالتوازي

    print(f"🚀 بدء سحب فتاوى من {BASE_URL} مع معالجة مسبقة. الهدف: {TARGET_COUNT} (لاحظ: سيتم تخطي {len(existing_links)} روابط موجودة)")

    while current_url and total_scraped_count < TARGET_COUNT:
        print(f"\n-> جلب صفحة الفهرس: {current_url}")
        try:
            resp = session.get(current_url, timeout=REQUEST_TIMEOUT)
            if resp.status_code == 500:
                print(" خطأ 500 بالخادم — سأنقح للصفحة التالية بعد انتظار قصير")
                time.sleep(10)
                # محاولة القفز لصفحة رقمية التالية (إن كان هناك باراميتر page)
                parsed = urlparse(current_url)
                q = parse_qs(parsed.query)
                page_num = int(q.get('page', [1])[0]) + 1
                current_url = f"{BASE_URL}/fatawa?page={page_num}"
                continue
            if resp.status_code != 200:
                print(f" حالة HTTP {resp.status_code}. إيقاف أو تخطي.")
                break

            soup = BeautifulSoup(resp.content, 'html.parser')
            entries = soup.select('div.list-item.entry')

            if not entries:
                print("لا توجد مدخلات في الصفحة — قد يكون تغيرت بنية الموقع أو الوصول محظور.")
                break

            # اجمع روابط الصفحات من الفهرس ضمن هذه الصفحة
            page_urls = []
            parsed_entries = []
            for e in entries:
                parsed = parse_listing_entry(e)
                if not parsed:
                    continue
                if parsed['URL'] in existing_links:
                    continue
                page_urls.append(parsed['URL'])
                parsed_entries.append(parsed)

            # إذا لا روابط جديدة في هذه الصفحة -> إلاحق، أو انهي
            if not page_urls:
                print("ℹ️ لا روابط جديدة في هذه الصفحة — الانتقال للصفحة التالية إن وُجدت")
            else:
                # جلب الصفحات بالتوازي لمعالجة النصوص
                print(f"🔎 جلب {len(page_urls)} صفحة تفاصيل بالتوازي (max_workers={MAX_WORKERS})")
                with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                    futures = {executor.submit(fetch_and_preprocess, u): u for u in page_urls}
                    for fut in as_completed(futures):
                        url = futures[fut]
                        try:
                            detail_result = fut.result()
                            # ابني السجل النهائي عبر دمج معلومات من القائمة + نتيجة المعالجة التفصيلية
                            # ابحث عن parsed_entry الملائم
                            parsed_entry = next((p for p in parsed_entries if p['URL'] == url), {})
                            record = {
                                'ID': parsed_entry.get('ID', extract_fatwa_id(url)),
                                'URL': url,
                                'المفتي': parsed_entry.get('المفتي', ''),
                                'العنوان': parsed_entry.get('العنوان', detail_result.get('title', '')),
                                'التصنيف': parsed_entry.get('التصنيف', ''),
                                'نص السؤال': parsed_entry.get('نص السؤال', ''),
                                'نص الإجابة': parsed_entry.get('نص الإجابة', ''),
                                'المحتوى_المعالج': detail_result.get('content', ''),
                                'content_length': detail_result.get('content_length', 0),
                                'words_count': detail_result.get('words_count', 0),
                                'تاريخ_النشر': parsed_entry.get('تاريخ النشر', ''),
                                'المصدر': parsed_entry.get('المصدر', urlparse(BASE_URL).netloc),
                                'status': detail_result.get('status', 'unknown')
                            }
                            batch.append(record)
                            existing_links.add(url)
                            total_scraped_count += 1

                            # حفظ الدفعات دورياً
                            if len(batch) >= SAVE_BATCH_SIZE:
                                save_batch_to_csv_and_pickle(batch, batch_number)
                                batch_number += 1
                                batch = []

                            # تحقق هدف العدد
                            if total_scraped_count >= TARGET_COUNT:
                                print(f" تم الوصول للعدد الهدف: {TARGET_COUNT}")
                                break

                        except Exception as e:
                            print(f" خطأ عند معالجة نتيجة {url}: {e}")
                            continue

            # تتبع وجود زر تحميل المزيد (load more) أو ترقيم صفحات
            load_more = soup.select_one('a#load-more')
            if load_more and load_more.get('href') and total_scraped_count < TARGET_COUNT:
                current_url = urljoin(BASE_URL, load_more.get('href'))
                # تأخير قصير لتخفيف الحمل
                time.sleep(0.8)
            else:
                # حاول الذهاب للصفحة التالية عن طريق زيادة page param إن لم يوجد load-more
                parsed = urlparse(current_url)
                q = parse_qs(parsed.query)
                current_page = int(q.get('page', [1])[0])
                next_page = current_page + 1
                next_url = f"{BASE_URL}/fatawa?page={next_page}"
                # تحقق بسيط إن لم يتغير الرابط، عندها نوقف
                if next_url == current_url:
                    print(" لم نتمكن من إيجاد صفحة تالية. إنهاء.")
                    current_url = None
                else:
                    current_url = next_url
                    time.sleep(0.8)

        except requests.exceptions.RequestException as e:
            print(f" خطأ طلب HTTP: {e}. محاولة إعادة المحاولة بعد تأخير طفيف.")
            time.sleep(10)
            continue
        except Exception as e:
            print(f" خطأ غير متوقع: {e}. إنهاء.")
            break

    # حفظ أي بيانات متبقية
    if batch:
        save_batch_to_csv_and_pickle(batch, batch_number)

    print(f"\n✅ انتهت عملية السحب. إجمالي السجلات المضافة في الجلسة: {total_scraped_count}")

# -------- تشغيل --------
if __name__ == "__main__":
    scrape_islamway_with_preprocessing()

ملاحظة: سكربت يدعم حفظ Drive. في حال رغبت بذلك نفّذ drive.mount يدوياً ثم فعّل SAVE_TO_DRIVE=True
ℹ️ لم يتم العثور على ملف سابق. البدء من البداية.
🚀 بدء سحب فتاوى من https://ar.islamway.net مع معالجة مسبقة. الهدف: 22000 (لاحظ: سيتم تخطي 0 روابط موجودة)

-> جلب صفحة الفهرس: https://ar.islamway.net/fatawa
🔎 جلب 12 صفحة تفاصيل بالتوازي (max_workers=20)



-> جلب صفحة الفهرس: https://ar.islamway.net/fatawa?page=2&lid=2692840&lrank=2025-09-17%2012%3A26%3A40
🔎 جلب 12 صفحة تفاصيل بالتوازي (max_workers=20)



-> جلب صفحة الفهرس: https://ar.islamway.net/fatawa?page=3&lid=2686922&lrank=2025-08-11%2012%3A44%3A27
🔎 جلب 12 صفحة تفاصيل بالتوازي (max_workers=20)



-> جلب صفحة الفهرس: https://ar.islamway.net/fatawa?page=4&lid=2670664&lrank=2025-04-27%2023%3A41%3A46
🔎 جلب 12 صفحة تفاصيل بالتوازي (max_workers=20)



-> جلب صفحة الفهرس: https://ar.islamway.net/fatawa?page=5&lid=2509236&lrank=2025-03-17%2017%3A18%3A00
🔎 جلب 12 صفحة تفاصيل بالتوازي (max_workers=20)


...


-> جلب صفحة الفهرس: https://ar.islamway.net/fatawa?page=489&lid=1215661&lrank=2015-05-14%2000%3A00%3A00
🔎 جلب 6 صفحة تفاصيل بالتوازي (max_workers=20)

-> جلب صفحة الفهرس: https://ar.islamway.net/fatawa?page=490&lid=1215461&lrank=2015-05-13%2000%3A00%3A00
🔎 جلب 11 صفحة تفاصيل بالتوازي (max_workers=20)


✅ تم حفظ الدفعة #28: أضيف 200 سجل. (ملف CSV محدث ونسخة احتياطية: ./ultra_fast_fatwa_backup/batches/batch_00028_20251031_134804.pkl)

-> جلب صفحة الفهرس: https://ar.islamway.net/fatawa?page=491&lid=1215298&lrank=2015-05-12%2000%3A00%3A00
🔎 جلب 8 صفحة تفاصيل بالتوازي (max_workers=20)

-> جلب صفحة الفهرس: https://ar.islamway.net/fatawa?page=492&lid=1214823&lrank=2015-05-11%2000%3A00%3A00
🔎 جلب 12 صفحة تفاصيل بالتوازي (max_workers=20)



-> جلب صفحة الفهرس: https://ar.islamway.net/fatawa?page=493&lid=1214478&lrank=2015-05-08%2000%3A00%3A00
🔎 جلب 10 صفحة تفاصيل بالتوازي (max_workers=20)

-> جلب صفحة الفهرس: https://ar.islamway.net/fatawa?page=494&lid=1214198&lrank=2015-05-06%2000%3A00%3A00
🔎 جلب 10 صفحة تفاصيل بالتوازي (max_workers=20)

-> جلب صفحة الفهرس: https://ar.islamway.net/fatawa?page=495&lid=1214190&lrank=2015-05-06%2000%3A00%3A00
🔎 جلب 6 صفحة تفاصيل بالتوازي (max_workers=20)

-> جلب صفحة الفهرس: https://ar.islamway.net/fatawa?page=496&lid=1214092&lrank=2015-05-05%2000%3A00%3A00
🔎 جلب 11 صفحة تفاصيل بالتوازي (max_workers=20)



-> جلب صفحة الفهرس: https://ar.islamway.net/fatawa?page=497&lid=1214004&lrank=2015-05-04%2000%3A00%3A00
🔎 جلب 12 صفحة تفاصيل بالتوازي (max_workers=20)



-> جلب صفحة الفهرس: https://ar.islamway.net/fatawa?page=498&lid=1213762&lrank=2015-05-01%2000%3A00%3A00
🔎 جلب 10 صفحة تفاصيل بالتوازي (max_workers=20)

-> جلب صفحة الفهرس: https://ar.islamway.net/fatawa?page=499&lid=1213621&lrank=2015-04-30%2000%3A00%3A00
 خطأ 500 بالخادم — سأنقح للصفحة التالية بعد انتظار قصير

-> جلب صفحة الفهرس: https://ar.islamway.net/fatawa?page=500
ℹ️ لا روابط جديدة في هذه الصفحة — الانتقال للصفحة التالية إن وُجدت

-> جلب صفحة الفهرس: https://ar.islamway.net/fatawa?page=501&lid=2692840&lrank=2025-09-17%2012%3A26%3A40
ℹ️ لا روابط جديدة في هذه الصفحة — الانتقال للصفحة التالية إن وُجدت

-> جلب صفحة الفهرس: https://ar.islamway.net/fatawa?page=502&lid=2686922&lrank=2025-08-11%2012%3A44%3A27
ℹ️ لا روابط جديدة في هذه الصفحة — الانتقال للصفحة التالية إن وُجدت

-> جلب صفحة الفهرس: https://ar.islamway.net/fatawa?page=503&lid=2670664&lrank=2025-04-27%2023%3A41%3A46
ℹ️ لا روابط جديدة في هذه الصفحة — الانتقال للصفحة التالية إن وُجدت

-> جلب صفحة الفهرس: https://ar.islamway.ne